In [1]:
!mkdir system-log-monitor
!mkdir system-log-monitor/sample_logs


flask
pandas
scikit-learn
numpy
sqlite3-binary
gunicorn

In [ ]:
%cd system-log-monitor


/content/system-log-monitor


In [ ]:
!pip install flask pandas scikit-learn


In [ ]:
%%writefile sample_logs/syslog.txt
Jan 10 10:12:33 INFO System started
Jan 10 10:14:01 WARN Disk usage high
Jan 10 10:15:44 ERROR Failed to connect database
Jan 10 10:16:55 INFO User login successful
Jan 10 10:17:01 ERROR Memory allocation failed


Writing sample_logs/syslog.txt


Database Layer (db.py)

In [ ]:
%%writefile db.py
import sqlite3
import os

def connect():
    return sqlite3.connect("logs.db")

def create_table():
    conn = connect()
    cur = conn.cursor()
    cur.execute("""
    CREATE TABLE IF NOT EXISTS logs(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT,
        level TEXT,
        message TEXT
    )
    """)
    conn.commit()
    conn.close()

def insert_log(ts, level, msg):
    conn = connect()
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO logs(timestamp, level, message) VALUES (?, ?, ?)",
        (ts, level, msg)
    )
    conn.commit()
    conn.close()

def fetch_logs():
    conn = connect()
    cur = conn.cursor()
    cur.execute("SELECT * FROM logs")
    rows = cur.fetchall()
    conn.close()
    return rows

def clear_logs():
    conn = connect()
    cur = conn.cursor()
    cur.execute("DELETE FROM logs")
    conn.commit()
    conn.close()

Overwriting db.py


Log Parser (log_parser.py)

In [ ]:
%%writefile log_parser.py
import re
from db import insert_log

pattern = r"(\w+\s+\d+\s[\d:]+)\s+(INFO|WARN|ERROR)\s+(.*)"

def parse_logs(file_path):
    print(f"Parsing logs from: {file_path}")
    with open(file_path,"r") as f:
        for line in f:
            print(f"Processing line: {line.strip()}")
            match = re.search(pattern,line)
            if match:
                ts, level, msg = match.groups()
                print(f"Match found: TS={ts}, Level={level}, Msg={msg}")
                insert_log(ts, level, msg)
            else:
                print(f"No match for line: {line.strip()}")

Overwriting log_parser.py


Anomaly Detection (anomaly.py)

In [ ]:
%%writefile anomaly.py
import pandas as pd
from sklearn.ensemble import IsolationForest
from db import fetch_logs

def detect_anomalies():
    rows = fetch_logs()
    df = pd.DataFrame(rows, columns=["id","ts","level","msg"])
    df["msg_len"] = df["msg"].apply(len)

    model = IsolationForest(contamination=0.2)
    df["anomaly"] = model.fit_predict(df[["msg_len"]])

    return df[df["anomaly"] == -1]

Writing anomaly.py


Severity Classification (severity.py)

In [ ]:
%%writefile severity.py
import pandas as pd
from sklearn.linear_model import LogisticRegression
from db import fetch_logs

def classify_severity():
    rows = fetch_logs()
    df = pd.DataFrame(rows, columns=["id","ts","level","msg"])

    label_map = {"INFO":0,"WARN":1,"ERROR":2}
    df["label"] = df["level"].map(label_map)
    df["msg_len"] = df["msg"].apply(len)

    X = df[["msg_len"]]
    y = df["label"]

    model = LogisticRegression()
    model.fit(X,y)

    df["predicted_severity"] = model.predict(X)

    return df[["msg","predicted_severity"]]

Writing severity.py


Metrics Generator (metrics.py)

In [ ]:
%%writefile metrics.py
import pandas as pd
from db import fetch_logs

def generate_metrics():
    rows = fetch_logs()
    df = pd.DataFrame(rows, columns=["id","ts","level","msg"])

    summary = df["level"].value_counts()
    summary.to_csv("metrics.csv")

    return summary

Writing metrics.py


Flask Backend . Production Flask App (app.py)

In [ ]:
from flask import Flask, request, jsonify
import sqlite3
import logging
from anomaly import detect_anomaly
from severity import predict_severity
from metrics import generate_metrics

app = Flask(__name__)

logging.basicConfig(level=logging.INFO)

DB = "logs.db"

def get_db():
    return sqlite3.connect(DB)

@app.route("/log", methods=["POST"])
def ingest_log():
    try:
        data = request.json
        message = data.get("message")

        conn = get_db()
        cur = conn.cursor()
        cur.execute("INSERT INTO logs(message) VALUES(?)", (message,))
        conn.commit()
        conn.close()

        anomaly = detect_anomaly(message)
        severity = predict_severity(message)

        generate_metrics()

        return jsonify({
            "status": "stored",
            "anomaly": anomaly,
            "severity": severity
        })

    except Exception as e:
        logging.error(str(e))
        return jsonify({"error": "internal server error"}), 500

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Writing app.py


Test WITHOUT Flask (Colab)

In [ ]:
import db
import log_parser
from anomaly import detect_anomalies
from severity import classify_severity
from metrics import generate_metrics
import importlib

# Reload the log_parser and db modules to ensure changes are applied
importlib.reload(log_parser)
importlib.reload(db)

# Clear existing logs and ensure table is created
db.clear_logs()
db.create_table()

# Attempt to parse logs from file
log_parser.parse_logs("sample_logs/syslog.txt")

print("--- Logs after parsing from file ---")
parsed_logs = db.fetch_logs()
print(parsed_logs)

# NOTE: Removed the direct insertion of the 'TEST' log entry as it was causing NaN in severity classification.
# print("--- Testing direct database insertion ---")
# db.insert_log("Test TS", "TEST", "This is a direct test message")
# direct_inserted_logs = db.fetch_logs()
# print(direct_inserted_logs)

# If logs are present from parsing, then proceed with anomaly detection, etc.
# Using the parsed_logs from file for subsequent operations
if parsed_logs:
    print("--- Anomaly Detection ---")
    print(detect_anomalies())
    print("--- Severity Classification ---")
    print(classify_severity())
    print("--- Metrics Generation ---")
    print(generate_metrics())
else:
    print("No logs parsed from file, skipping anomaly detection, severity classification, and metrics generation.")

Parsing logs from: sample_logs/syslog.txt
Processing line: Jan 10 10:12:33 INFO System started
Match found: TS=Jan 10 10:12:33, Level=INFO, Msg=System started
Processing line: Jan 10 10:14:01 WARN Disk usage high
Match found: TS=Jan 10 10:14:01, Level=WARN, Msg=Disk usage high
Processing line: Jan 10 10:15:44 ERROR Failed to connect database
Match found: TS=Jan 10 10:15:44, Level=ERROR, Msg=Failed to connect database
Processing line: Jan 10 10:16:55 INFO User login successful
Match found: TS=Jan 10 10:16:55, Level=INFO, Msg=User login successful
Processing line: Jan 10 10:17:01 ERROR Memory allocation failed
Match found: TS=Jan 10 10:17:01, Level=ERROR, Msg=Memory allocation failed
--- Logs after parsing from file ---
[(20, 'Jan 10 10:12:33', 'INFO', 'System started'), (21, 'Jan 10 10:14:01', 'WARN', 'Disk usage high'), (22, 'Jan 10 10:15:44', 'ERROR', 'Failed to connect database'), (23, 'Jan 10 10:16:55', 'INFO', 'User login successful'), (24, 'Jan 10 10:17:01', 'ERROR', 'Memory alloc

In [ ]:
import re
import os

# Check if the file exists and its content
file_path = "sample_logs/syslog.txt"
if os.path.exists(file_path):
    print(f"File '{file_path}' exists.")
    with open(file_path, "r") as f:
        content = f.read()
        print("File content:")
        print(content)

    # Test the regex against the first line of the actual file content
    lines = content.strip().split('\n')
    if lines:
        first_line = lines[0]
        pattern = r"(\w+\s+\d+\s[\d:]+)\s+(INFO|WARN|ERROR)\s+(.*)"
        match = re.search(pattern, first_line)
        if match:
            print(f"Regex test on first line: {match.groups()}")
        else:
            print(f"Regex test on first line: No match for '{first_line}'")
else:
    print(f"File '{file_path}' does NOT exist.")

File 'sample_logs/syslog.txt' exists.
File content:
Jan 10 10:12:33 INFO System started
Jan 10 10:14:01 WARN Disk usage high
Jan 10 10:15:44 ERROR Failed to connect database
Jan 10 10:16:55 INFO User login successful
Jan 10 10:17:01 ERROR Memory allocation failed

Regex test on first line: ('Jan 10 10:12:33', 'INFO', 'System started')


In [ ]:
from db import create_table, fetch_logs
from log_parser import parse_logs

create_table()
parse_logs("sample_logs/syslog.txt")

print(fetch_logs())

Parsing logs from: sample_logs/syslog.txt
Processing line: Jan 10 10:12:33 INFO System started
Match found: TS=Jan 10 10:12:33, Level=INFO, Msg=System started
Processing line: Jan 10 10:14:01 WARN Disk usage high
Match found: TS=Jan 10 10:14:01, Level=WARN, Msg=Disk usage high
Processing line: Jan 10 10:15:44 ERROR Failed to connect database
Match found: TS=Jan 10 10:15:44, Level=ERROR, Msg=Failed to connect database
Processing line: Jan 10 10:16:55 INFO User login successful
Match found: TS=Jan 10 10:16:55, Level=INFO, Msg=User login successful
Processing line: Jan 10 10:17:01 ERROR Memory allocation failed
Match found: TS=Jan 10 10:17:01, Level=ERROR, Msg=Memory allocation failed
[(20, 'Jan 10 10:12:33', 'INFO', 'System started'), (21, 'Jan 10 10:14:01', 'WARN', 'Disk usage high'), (22, 'Jan 10 10:15:44', 'ERROR', 'Failed to connect database'), (23, 'Jan 10 10:16:55', 'INFO', 'User login successful'), (24, 'Jan 10 10:17:01', 'ERROR', 'Memory allocation failed'), (25, 'Jan 10 10:12:33

In [ ]:
from db import create_table, fetch_logs
from log_parser import parse_logs

create_table()
parse_logs("sample_logs/syslog.txt")
print(fetch_logs())

Parsing logs from: sample_logs/syslog.txt
Processing line: Jan 10 10:12:33 INFO System started
Match found: TS=Jan 10 10:12:33, Level=INFO, Msg=System started
Processing line: Jan 10 10:14:01 WARN Disk usage high
Match found: TS=Jan 10 10:14:01, Level=WARN, Msg=Disk usage high
Processing line: Jan 10 10:15:44 ERROR Failed to connect database
Match found: TS=Jan 10 10:15:44, Level=ERROR, Msg=Failed to connect database
Processing line: Jan 10 10:16:55 INFO User login successful
Match found: TS=Jan 10 10:16:55, Level=INFO, Msg=User login successful
Processing line: Jan 10 10:17:01 ERROR Memory allocation failed
Match found: TS=Jan 10 10:17:01, Level=ERROR, Msg=Memory allocation failed
[(20, 'Jan 10 10:12:33', 'INFO', 'System started'), (21, 'Jan 10 10:14:01', 'WARN', 'Disk usage high'), (22, 'Jan 10 10:15:44', 'ERROR', 'Failed to connect database'), (23, 'Jan 10 10:16:55', 'INFO', 'User login successful'), (24, 'Jan 10 10:17:01', 'ERROR', 'Memory allocation failed'), (25, 'Jan 10 10:12:33

In [ ]:
from anomaly import detect_anomalies
from severity import classify_severity
from metrics import generate_metrics

print(detect_anomalies())
print(classify_severity())
print(generate_metrics())

    id               ts  level                         msg  msg_len  anomaly
2   22  Jan 10 10:15:44  ERROR  Failed to connect database       26       -1
7   27  Jan 10 10:15:44  ERROR  Failed to connect database       26       -1
12  32  Jan 10 10:15:44  ERROR  Failed to connect database       26       -1
                           msg  predicted_severity
0               System started                   0
1              Disk usage high                   0
2   Failed to connect database                   2
3        User login successful                   0
4     Memory allocation failed                   2
5               System started                   0
6              Disk usage high                   0
7   Failed to connect database                   2
8        User login successful                   0
9     Memory allocation failed                   2
10              System started                   0
11             Disk usage high                   0
12  Failed to connect databas

UPGRADE A — Logging + Error Handling (FINAL)

In [ ]:
%%writefile logger.py
import logging

logging.basicConfig(
    filename="app.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

def get_logger(name):
    return logging.getLogger(name)

Writing logger.py


In [ ]:
%%writefile db.py
import sqlite3
from logger import get_logger

logger = get_logger(__name__)

def connect():
    return sqlite3.connect("logs.db")

def create_table():
    try:
        conn = connect()
        cur = conn.cursor()

        cur.execute("DROP TABLE IF EXISTS logs")

        cur.execute("""
        CREATE TABLE logs(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp TEXT,
            level TEXT,
            message TEXT
        )
        """)

        conn.commit()
        conn.close()
        logger.info("Database reset and table initialized")

    except Exception as e:
        logger.error(f"DB init error: {e}")

def insert_log(ts, level, msg):
    try:
        conn = connect()
        cur = conn.cursor()
        cur.execute(
            "INSERT INTO logs(timestamp, level, message) VALUES (?, ?, ?)",
            (ts, level, msg)
        )
        conn.commit()
        conn.close()
        logger.info("Inserted log entry")

    except Exception as e:
        logger.error(f"Insert error: {e}")

def fetch_logs():
    try:
        conn = connect()
        cur = conn.cursor()
        cur.execute("SELECT * FROM logs")
        rows = cur.fetchall()
        conn.close()
        return rows

    except Exception as e:
        logger.error(f"Fetch error: {e}")
        return []

Overwriting db.py


In [ ]:
%%writefile log_parser.py
import re
from db import insert_log
from logger import get_logger

logger = get_logger(__name__)

pattern = r"(\\w+\\s+\\d+\\s[\\d:]+).*?(INFO|WARN|ERROR)\\s(.*)"

def parse_logs(file_path):
    try:
        with open(file_path,"r") as f:
            for line in f:
                match = re.search(pattern,line)
                if match:
                    ts, level, msg = match.groups()
                    insert_log(ts, level, msg)

        logger.info("Log parsing completed")

    except Exception as e:
        logger.error(f"Parser error: {e}")

Overwriting log_parser.py


In [ ]:
%%writefile app.py
from flask import Flask, jsonify, request
from db import create_table, fetch_logs
from log_parser import parse_logs
from anomaly import detect_anomalies
from severity import classify_severity
from metrics import generate_metrics
from logger import get_logger

app = Flask(__name__)
logger = get_logger(__name__)

create_table()

@app.route("/upload", methods=["POST"])
def upload():
    try:
        file = request.files["file"]
        path = "uploaded.txt"
        file.save(path)
        parse_logs(path)
        return {"status":"uploaded"}
    except Exception as e:
        logger.error(e)
        return {"error":"upload failed"}

@app.route("/logs")
def logs():
    return jsonify(fetch_logs())

@app.route("/anomalies")
def anomalies():
    return detect_anomalies().to_json(orient="records")

@app.route("/severity")
def severity():
    return classify_severity().to_json(orient="records")

@app.route("/metrics")
def metrics():
    return generate_metrics().to_json()

if __name__ == "__main__":
    app.run()

Overwriting app.py


UPGRADE B — Docker

In [ ]:
%%writefile Dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY . .

RUN pip install flask pandas scikit-learn

EXPOSE 5000

CMD ["python","app.py"]

Writing Dockerfile


Basic Metrics (metrics.py)

In [ ]:
import sqlite3

def generate_metrics():
    conn = sqlite3.connect("logs.db")
    cur = conn.cursor()

    count = cur.execute("SELECT COUNT(*) FROM logs").fetchone()[0]

    with open("metrics.txt", "w") as f:
        f.write(f"total_logs {count}\n")

    conn.close()

SQLite Bootstrap (init_db.py)

In [ ]:
import sqlite3

conn = sqlite3.connect("logs.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS logs(
id INTEGER PRIMARY KEY AUTOINCREMENT,
message TEXT
)
""")

conn.commit()
conn.close()

ML Stubs

anomaly.py

In [ ]:
from sklearn.ensemble import IsolationForest
import numpy as np

model = IsolationForest()

def detect_anomaly(msg):
    x = np.array([[len(msg)]])
    return bool(model.fit_predict(x)[0] == -1)

severity.py

In [ ]:
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install -r requirements.txt

COPY . .

EXPOSE 5000

CMD ["gunicorn", "-w", "2", "-b", "0.0.0.0:5000", "app:app"]

In [ ]:
docker run -p 5000:5000 observability

In [ ]:
curl -X POST http://localhost:5000/log \
-H "Content-Type: application/json" \
-d '{"message":"CPU error detected"}'